# Project Phase 1: Pre-processing (MICE 補完 & 全会合集約可視化)

このノートブックでは、MICEによる全BOJ会合金利の補完結果を一枚のグラフに集約して描画し、全系列の定常性を分数階差で確認します。

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

%load_ext autoreload
%autoreload 2

from src.processing import load_and_clean_data
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

%matplotlib inline
sns.set(style='whitegrid')
plt.rcParams['font.family'] = 'AppleGothic'

## 1. データのクレンジング実行と全生データの保持

In [ ]:
excel_path = '../data/BOJ_data.xlsx'
meeting_csv = '../data/BOJ_meeting_history.csv'
boj_cols = ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8']

df = load_and_clean_data(excel_path, meeting_csv)
df.to_csv('../data/cleaned_data.csv', index=False)
print(f'Data size after cleaning: {df.shape}')

## 2. 全BOJ会合期待金利の補完後の推移 (M1-M8 集約プロット)
MICE補完後のすべての会合期待金利を一望します。

In [ ]:
plt.figure(figsize=(15, 8))
plt.step(df['日付'], df['Actual_Policy_Rate'], where='post', color='black', lw=4, label='Actual Policy Rate', alpha=0.9)
colors = plt.cm.viridis(np.linspace(0, 1, len(boj_cols)))
for i, m in enumerate(boj_cols):
    plt.plot(df['日付'], df[m], label=m, color=colors[i], alpha=0.7)

plt.title('All BOJ Meeting Expectation Rates (M1-M8) - MICE Imputed', fontsize=16)
plt.ylabel('Rate (%)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()

## 3. 全BOJ会合期待金利の分数階差 (d=0.4) 集約プロット
すべての分数階差系列を重ねて表示し、定常性を一括で確認します。

In [ ]:
def frac_diff(series, d, window=50):
    w = [1.0]
    for k in range(1, window):
        w.append(-w[-1] * (d - k + 1) / k)
    weights = np.array(w[::-1])
    res = series.rolling(window=window).apply(lambda x: np.sum(x * weights), raw=True)
    return res

plt.figure(figsize=(15, 8))
for i, m in enumerate(boj_cols):
    fd_series = frac_diff(df[m], 0.4)
    plt.plot(df['日付'], fd_series, label=f'{m} d=0.4', color=colors[i], alpha=0.7)

plt.title('All Fractionally Differentiated BOJ Rates (d=0.4)', fontsize=16)
plt.ylabel('Differentiated Value')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()